In [2]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [3]:
import pandas as pd
from tqdm import tqdm
from utils import combined_approaches as ca
from utils import global_strategies as gs
from utils import label_based_measures as lbm
from utils import local_single_attribute as lsa
from utils import similarity_structures as ss
from utils import top_down_data_structures as tdds
from utils import value_overlap as vo
from utils import evaluation as eval

<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
# Esercizio Top-Down : Camera

In [4]:
src_links = [
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/CameraS1_B.csv',
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/CameraS2_B.csv',
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/CameraS3_B.csv',
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/CameraS4_B.csv',
]
SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }
GlobalSchema=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/GlobalClass_Bis.csv').astype(str)
GoldStandard=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/GoldStandardFull.csv').astype(str)
print(GlobalSchema.columns.to_list())
tdds.to_GMM(GoldStandard)

['brand', 'image_format', 'auto_focus_beam', 'auto_focus_mode', 'exposure_mode', 'battery_chemistry']


SOURCE,S1,S2,S3,S4
GAT,,,,
auto_focus_beam,[af assist beam],[],[],[af illuminator]
auto_focus_mode,"[autofocus af, af modes, focusarea selection]",[auto focusing af modes],"[auto focus, focus]",[autofocus]
battery_chemistry,[batteries],"[battery technology, battery type]",[battery type],[battery]
brand,[],[manufacturer],[brand],"[brand, product name]"
exposure_mode,[exposure control],"[light exposure modes, light exposure control]",[],[exposure modes]
image_format,"[file formats, still image type]",[image formats supported],[image format],[image format]


In [5]:
eval.AnalisiGlobalMatchTable(GMT=GoldStandard, Sources=SOURCES)

1) Le seguenti SOURCE di GMT non sono definite in Sources: []
2) I seguenti SLAT di GMT non sono definiti in Sources: ['S2_manufacturer']
3) GAT mappati da una sola SOURCE: []
4) GAT mappati in più LAT (per ciascuna SOURCE):
   SOURCE: S1, GAT: auto_focus_mode, LAT diversi: 3
   SOURCE: S1, GAT: image_format, LAT diversi: 2
   SOURCE: S2, GAT: battery_chemistry, LAT diversi: 2
   SOURCE: S2, GAT: exposure_mode, LAT diversi: 2
   SOURCE: S3, GAT: auto_focus_mode, LAT diversi: 2
   SOURCE: S4, GAT: brand, LAT diversi: 2
5) LAT mappati in più GAT (per ciascuna SOURCE):


In [6]:
ValutazioneMatchTable = pd.DataFrame(columns=['MT', 'TP', 'FP', 'FN', 'P', 'R', 'F'])

In [28]:
def CalcoloMatchingTable(TableL:pd.DataFrame,TableR:pd.DataFrame):

# Matching Methods: Qui si definiscono i Base Matcher da utilizzare; se ne possono aggiungere altri    
        SimTableA = lbm.levenshtein_label_based_similarity(TableL, TableR)
        SimTableB = lbm.jaro_label_based_similarity(TableL, TableR)
        # SimTableB = vo.value_overlap_sim(GlobalSchema, Sources[y])
        SimTableC=  vo.value_overlap_simjoin_jaccard(TableL, TableR, 0.5)

# Combined Approaches: qui si definisce il combiner; se ne possono aggiungere altri    
        #SimTable = ca.avg_sim_table([SimTableA,SimTableB, SimTableC])
        #SimTable = ca.min_sim_table([SimTableA,SimTableB, SimTableC])
        SimTable = ca.max_sim_table([SimTableA,SimTableB, SimTableC])

        # Weighted-sum
        #SimTable = ca.Weighted_sum([SimTableA,SimTableB,SimTableC], [.3,.4,.3] )
        
# Generating Correspondences: dalla tabella di similarità alle corrispondenze

### Local Single Attribute Strategies:
        MatchTable= lsa.thresholding(SimTable, 0.2)

        #MatchTable = lsa.top_K(SimTable,1,'A')
        MatchTable = lsa.top_K(MatchTable,1,'B')

        
### Global Mapping (si usano solo questi due metodi)
        #MatchTable = gs.stable_marriage(MatchTable)
        #MatchTable = gs.simmetric_best_match(MatchTable)

        return MatchTable

In [29]:
def CalcoloGlobalMatchingTable(Sources, GlobalSchema:pd.DataFrame):
    GlobalMatchingTable = pd.DataFrame(columns=['GAT','SOURCE','LAT','SLAT','sim'])
    for y in tqdm(Sources.keys()):
        MatchTable = CalcoloMatchingTable(GlobalSchema, Sources[y])
        
        MatchTable.columns = ['GAT','LAT','sim']
        MatchTable['SOURCE'] = str(y)
        MatchTable['SLAT'] = MatchTable['SOURCE']+'_'+MatchTable['LAT']
        GlobalMatchingTable = GlobalMatchingTable.append(MatchTable, sort=False)

    return GlobalMatchingTable

In [30]:
GMTcalcolata=CalcoloGlobalMatchingTable(SOURCES, GlobalSchema)
tdds.to_GMM(GMTcalcolata)

100%|██████████| 4/4 [00:02<00:00,  1.80it/s]


SOURCE,S1,S2,S3,S4
GAT,,,,
auto_focus_beam,[af assist beam],[],[],[af illuminator]
auto_focus_mode,"[af modes, autofocus af, focusarea selection]",[auto focusing af modes],"[auto focus, focus]",[autofocus]
battery_chemistry,[batteries],"[battery technology, battery type]",[battery type],[battery]
brand,[],[],[brand],"[brand, product name]"
exposure_mode,[exposure control],"[light exposure control, light exposure modes]",[],[exposure modes]
image_format,"[file formats, still image type]",[image formats supported],[image format],[image format]


In [32]:
X=eval.Valuta(GoldStandard[['GAT', 'SLAT']],GMTcalcolata[['GAT', 'SLAT']])
X

,MT,TP,FP,FN,P,R,F
0,26,26,0,1,1.0,0.963,0.9811


In [33]:
ValutazioneMatchTable = ValutazioneMatchTable.append(X).rename(index={0: "Label-Instance-Max-0.4-Top1-NoGlobal11"})
ValutazioneMatchTable

,MT,TP,FP,FN,P,R,F
Label-Instance-Avg-0.4-Top1-NoGlobal11,26,26,0,1,1.0000,0.9630,0.9811
Label-Instance-Avg-0.4-Top1-NoGlobal11,24,19,5,8,0.7917,0.7037,0.7451
Label-Instance-Max-0.4-Top1-NoGlobal11,26,26,0,1,1.0000,0.9630,0.9811
